# Leaflet cluster map of talk locations

Assuming you are working in a Linux or Windows Subsystem for Linux environment, you may need to install some dependencies. Assuming a clean installation, the following will be needed:

```bash
sudo apt install jupyter
sudo apt install python3-pip
pip install python-frontmatter getorg --upgrade
```

After which you can run this from the `_talks/` directory, via:

```bash
 jupyter nbconvert --to notebook --execute talkmap.ipynb --output talkmap_out.ipynb
```
 
The `_talks/` directory contains `.md` files of all your talks. This scrapes the location YAML field from each `.md` file, geolocates it with `geopy/Nominatim`, and uses the `getorg` library to output data, HTML, and Javascript for a standalone cluster map.

In [ ]:
# Dependencies are installed by the workflow (.github/workflows/scrape_talks.yml).
# To run locally: pip install python-frontmatter geopy getorg ipyleaflet ipywidgets
import re
import time
import glob
import frontmatter
import getorg
from geopy import Nominatim
from geopy.exc import GeocoderTimedOut, GeocoderServiceError

In [ ]:
# Collect the Markdown files
g = glob.glob("_talks/*.md")

In [ ]:
# Set the default timeout, in seconds
TIMEOUT = 5

# Prepare to geolocate
geocoder = Nominatim(user_agent="pastelbelem8.github.io talkmap")
location_dict = {}
location = ""
permalink = ""
title = ""

In the event that this times out with an error, double check to make sure that the location is can be properly geolocated.

In [ ]:
# Perform geolocation
for file in g:
    # Read the file
    data = frontmatter.load(file).to_dict()

    # Press on if the location is not present
    if 'location' not in data:
        continue

    # Prepare the description (use .get so a missing title/venue does not crash the run)
    title = data.get('title', '').strip()
    venue = data.get('venue', '').strip()
    location = data['location'].strip()
    description = f"{title}<br />{venue}; {location}"

    # Strip parenthetical qualifiers like "(remote)" so the geocoder can resolve the place
    query = re.sub(r"\s*\([^)]*\)", "", location).strip()

    # Geocode the location and report the status
    try:
        result = geocoder.geocode(query, timeout=TIMEOUT)
        if result is None:
            print(f"Warning: no geocoding match for {query!r} (from {location!r}); skipping")
            continue
        location_dict[description] = result
        print(description, result)
        time.sleep(1)  # respect Nominatim's 1 request/second usage policy
    except (ValueError, GeocoderTimedOut, GeocoderServiceError) as ex:
        print(f"Error: geocode failed on input {query!r} with message {ex}")
    except Exception as ex:
        print(f"An unhandled exception occurred while processing input {query!r} with message {ex}")

In [ ]:
# Save the map
m = getorg.orgmap.create_map_obj()
getorg.orgmap.output_html_cluster_map(location_dict, folder_name="talkmap", hashed_usernames=False)